# درس ۱۳ - حافظه عامل با نمودارهای دانش Cognee


## Setup

This notebook demonstrates how to build an intelligent **coding assistant** with persistent memory using [**Cognee**](https://www.cognee.ai/) knowledge graphs and the **Microsoft Agent Framework** (MAF).

Cognee transforms unstructured text into a structured, queryable knowledge graph backed by vector embeddings — giving your agent a rich, relationship-aware long-term memory.

### What You'll Learn
1. **Build Knowledge Graphs**: Transform developer profiles and best practices into structured, queryable knowledge.
2. **Integrate Cognee with MAF**: Use `@tool` functions to let an MAF agent query Cognee's knowledge graph.
3. **Session-Aware Conversations**: Maintain context across multiple questions in the same session.
4. **Long-Term Memory**: Persist important knowledge across sessions and retrieve it in new conversations.

### Prerequisites
- Python 3.9+
- Redis running locally (`docker run -d -p 6379:6379 redis`) for session management
- An LLM API key (e.g. OpenAI) — set `LLM_API_KEY` in `.env`
- `CACHING=true` in `.env` (required for Cognee sessions)
- A Microsoft Foundry project with a deployed chat model
- `AZURE_AI_PROJECT_ENDPOINT` and `AZURE_AI_MODEL_DEPLOYMENT_NAME` in `.env`
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## انواع حافظه عامل

این دفترچه همان سه نوع حافظه از دفترچه درس ۱۳ اصلی را بررسی می‌کند، اما از Cognee به عنوان پشتیبان حافظه بلندمدت استفاده می‌کند:

| نوع حافظه | مکانیزم | طول عمر |
|---|---|---|
| **کاری** | `agent.create_session()` (MAF) | یک گفتگو |
| **کوته‌مدت** | کش جلسه Cognee (Redis) | یک جلسه |
| **بلندمدت** | گراف دانش Cognee + بردارها | دائمی |

### معماری حافظه Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## آماده‌سازی فضای ذخیره‌سازی Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## بخش 1 — ساخت پایگاه دانش

ما سه نوع داده را وارد می‌کنیم تا یک پایگاه دانش جامع برای دستیار برنامه‌نویسی خود ایجاد کنیم:

1. **پروفایل توسعه‌دهنده** — تخصص شخصی و پیش‌زمینه فنی
2. **بهترین شیوه‌های پایتون** — ذن پایتون با راهنمایی‌های عملی
3. **گفتگوهای تاریخی** — جلسات سوال و جواب گذشته بین توسعه‌دهندگان و دستیاران هوش مصنوعی


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## تجسم گراف دانش

Cognee می‌تواند یک تصویر تعاملی HTML از موجودیت‌ها و روابطی که استخراج کرده است را نمایش دهد.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## غنی‌سازی حافظه با Memify

`memify()` گراف دانش را تجزیه و تحلیل کرده و قوانین هوشمندانه‌ای تولید می‌کند — که الگوها، بهترین شیوه‌ها و روابط بین مفاهیم را شناسایی می‌کند.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## بخش ۲ — عامل MAF با ابزارهای Cognee

اکنون یک عامل MAF ایجاد می‌کنیم که می‌تواند از طریق توابع `@tool` به گراف دانش Cognee پرس‌وجو کند. این به عامل اجازه می‌دهد تا از تمام قدرت جستجوی معنایی آگاه به گراف استفاده کند و در عین حال زمینه مکالمه‌ای را در طول جلسات حفظ کند.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## حافظه کاری با جلسات

 `AgentSession` (ایجاد شده از طریق `agent.create_session()`) حافظه کاری را در یک جلسه فراهم می‌کند. عامل می‌تواند به پیام‌های قبلی مراجعه کند و همچنین گراف دانش بلندمدت Cognee را جستجو نماید.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## جلسه جدید — حافظه بلندمدت باقی می‌ماند

شروع یک جلسه تازه، حافظه کاری را پاک می‌کند، اما نمودار دانش Cognee همچنان در دسترس است. عامل می‌تواند همان دانش بلندمدت را در یک مکالمه کاملاً جدید بازیابی کند.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## خلاصه

در این دفترچه، شما یک دستیار برنامه‌نویسی ساختید که **حافظه کاری MAF** (`agent.create_session()`) را با **گراف دانش بلندمدت Cognee** ترکیب می‌کند.

### آنچه آموخته‌اید
1. **ساخت گراف دانش**: Cognee متن‌های بدون ساختار را دریافت و گراف + حافظه وکتوری می‌سازد.
2. **غنی‌سازی گراف با memify**: حقایق استخراج‌شده و روابط غنی‌تر بر اساس گراف موجود شما.
3. **یکپارچه‌سازی MAF + Cognee**: توابع `@tool` به عامل MAF اجازه می‌دهند به طور طبیعی از گراف Cognee پرس‌وجو کند.
4. **حافظه کاری + حافظه بلندمدت**: `AgentSession` (از طریق `agent.create_session()`) زمینه جلسه را فراهم می‌کند در حالی که Cognee دانش پایدار ارائه می‌دهد.
5. **جستجوی فیلتر شده با NodeSets**: هدف‌گیری زیرمجموعه‌های خاصی از گراف دانش (مثلاً فقط اصول).

### نکات کلیدی
- **Cognee** متن خام را به حافظه‌ای ساختارمند و آگاه به روابط تبدیل می‌کند — قدرت بیشتری نسبت به یک فروشگاه وکتوری ساده دارد.
- **توابع `@tool`** به‌خوبی پل ارتباطی بین عوامل MAF و سیستم‌های دانش خارجی هستند.
- **`AgentSession`** (از طریق `agent.create_session()`) زمینه هر مکالمه را از دانش بلندمدت جدا نگه می‌دارد.
- همان گراف دانش به چندین جلسه و عامل خدمت می‌کند.

### کاربردهای دنیای واقعی
- **همیارهای توسعه‌دهنده**: بازبینی کد، تحلیل حوادث، دستیارهای معماری
- **همیارهای مشتری‌محور**: نمایندگان پشتیبانی روی اسناد محصول، سوالات متداول، و یادداشت‌های CRM
- **همیارهای کارشناسی داخلی**: دستیارهای سیاست‌گذاری، حقوقی، یا امنیت که بر روی دستورالعمل‌ها استدلال می‌کنند
- **لایه‌های داده متحد**: ترکیب داده‌های ساختارمند و بدون ساختار در یک گراف قابل جستجو

### گام‌های بعدی
- آزمایش آگاهی زمانی در Cognee
- تعریف یک آنتولوژی OWL برای کیفیت گراف حوزه‌محور
- افزودن حلقه‌های بازخورد کاربری برای بهبود بازیابی در طول زمان
- توسعه به سیستم‌های چندعاملی که همان لایه حافظه Cognee را به اشتراک می‌گذارند


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
